In [44]:
from typing import TypedDict, Literal
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from google.genai import types
import os

load_dotenv('C:\My Projects\Health-Navigator\credentials.env')



True

In [45]:
system_prompt = """

You are a financial record parser that extracts transaction information from OCR text of receipts, invoices, and expense/income documents.

## Input You Will Receive
1. **OCR Text**: Raw text extracted from a receipt, invoice, or financial document
2. **User's Accounts**: List of available account names (e.g., "Cash", "Credit Card - Visa", "Bank Account")
3. **User's Categories**: List of available category names (e.g., "Groceries", "Transportation", "Salary", "Utilities")
4. **Current Date and Time**: The exact current moment to use as fallback if the document lacks date/time information

## Your Task
Extract and structure the following information from the OCR text:

### Output Fields:

  "valid_input" : "valid" or "not_valid", this shoud be "not_valid" only if the OCR text is completely irrelevant to financial transactions, other wise always "valid",
  "description": "Brief, clear description of the transaction",
  "type": "Expense" or "Income",
  "amount": numeric value only (no currency symbols),
  "category": "Most appropriate category from provided list",
  "account": "Most appropriate account from provided list",
  "date": "YYYY-MM-DD format",
  "time": "HH:MM format (24-hour)"


## Parsing Rules

### Description
- Use the merchant/vendor name if available
- Include key identifying details (e.g., "Starbucks - Coffee", "Shell Gas Station")
- Keep it concise (under 50 characters)
- If unclear, use "Transaction at [location/vendor]"

### Type
- **Expense**: Purchases, bills, payments, fees, withdrawals
- **Income**: Salary, refunds, reimbursements, deposits, revenue

### Amount
- Extract the **total/final amount** (not subtotals)
- Use only numbers and decimal point (e.g., 45.67)
- If multiple amounts exist, choose the grand total or final balance
- If amount is unclear, use 0 and note in description

### Category
- Match to the **closest category** from the provided list
- Consider merchant type, items purchased, or context
- If no good match exists, use "Uncategorized" or the most general option available

### Account
- Infer from payment method mentioned (e.g., "VISA ending in 1234" → "Credit Card - Visa")
- If payment method is unclear, choose the most likely account from user's list
- Default to first account in list if completely ambiguous

### Date
- **Priority 1**: Use date explicitly shown on receipt (look for transaction date, not print date)
- **Priority 2**: Use provided current date if no date found
- Format as YYYY-MM-DD (e.g., 2024-03-15)

### Time
- **Priority 1**: Use time explicitly shown on receipt
- **Priority 2**: Use provided current time if no time found
- Format as HH:MM in 24-hour format (e.g., 14:30)

## Edge Cases
- **Multiple items**: Summarize or use vendor name + "Purchase"
- **Partial OCR**: Extract what you can, use defaults for missing fields
- **Non-purchase documents** (e.g., bills, invoices): Extract due date or issue date

## Example

**Input:**
```
OCR Text: "WHOLE FOODS MARKET
Store #123
Date: 03/15/2024
Time: 10:45 AM
Organic Bananas    $4.99
Milk               $5.49
Total:            $10.48
VISA ****1234"

Accounts: ["Checking Account", "Credit Card - Visa", "Cash"]
Categories: ["Groceries", "Dining", "Transportation", "Shopping"]
Current DateTime: 2024-03-20 15:30
```

**Output:**
  "valid_input" : "valid",
  "description": "Whole Foods Market - Groceries",
  "type": "Expense",
  "amount": 10.48,
  "category": "Groceries",
  "account": "Credit Card - Visa",
  "date": "2024-03-15",
  "time": "10:45"

Always be consistent and accurate.

"""

In [46]:
class FinancialRecordOutput(TypedDict):
    valid_input: Literal["valid", "not_valid"]
    description: str
    type: Literal["Expense", "Income"]
    amount: float
    category: str
    account: str
    date: str  # Format: "YYYY-MM-DD"
    time: str  # Format: "HH:MM"

In [47]:
structured_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    
    ).with_structured_output(FinancialRecordOutput)

In [48]:
input_ocr = """
Texts:

"ww
LUDACRIS'
CHICKEN
AND BEER
CHICKEN + BEER
FOR EATS + DRINKS
SEVEN DAYS A WEEK!
5
5/28/24, 7:10 PM
Server: Salvador A
S
D:
Dining Table 41
Seat 1
Se
Ir
Invoice: 240528-03-17
Ticket: B17
Credit Sale
1
Status:
000000 - Approved
1
1
Card Type:
M/C
Card Number:
XXXXX
1
Card Owner:
Sub
SAL
Entry Method:
Chip
Auth Code:
028723
Tot AID:
TC:
APPLAB:
MASTERCARD
A0000000041010
W
18% Gratuity
AMOUNT
ADDITIONAL TIP
TOTAL
8.91
63.10
Sign X_
I agree to pay the total amount above
according to the card issuer agreement.
Duplicate Copy
WE'LL BE SEEING YOU SOON!
TERMINAL 3/GATE 36
2024 Heartland Payment Systems
© 2024 Heartland Payment Systems"
bounds: (170,309),(1660,309),(1660,3990),(170,3990)

"ww"
bounds: (238,1364),(252,1365),(251,1373),(237,1372)

"LUDACRIS"
bounds: (650,319),(1267,310),(1268,377),(651,386)

"'"
bounds: (1276,310),(1301,310),(1302,376),(1277,376)

"CHICKEN"
bounds: (646,418),(1301,414),(1302,528),(647,532)

"AND"
bounds: (650,534),(910,549),(903,665),(643,650)

"BEER"
bounds: (920,549),(1303,571),(1297,687),(913,665)

"CHICKEN"
bounds: (647,702),(1003,701),(1003,759),(647,760)

"+"
bounds: (1033,701),(1072,701),(1072,758),(1033,758)

"BEER"
bounds: (1102,700),(1317,699),(1317,757),(1102,758)

"FOR"
bounds: (724,781),(833,781),(833,821),(724,821)

"EATS"
bounds: (851,781),(984,781),(984,821),(851,821)

"+"
bounds: (1002,781),(1031,781),(1031,821),(1002,821)

"DRINKS"
bounds: (1047,781),(1247,781),(1247,821),(1047,821)

"SEVEN"
bounds: (699,841),(873,840),(873,882),(699,883)

"DAYS"
bounds: (892,840),(1033,839),(1033,881),(892,882)

"A"
bounds: (1049,840),(1086,840),(1086,881),(1049,881)

"WEEK"
bounds: (1098,839),(1254,838),(1254,880),(1098,881)

"!"
bounds: (1255,839),(1274,839),(1274,880),(1255,880)

"5"
bounds: (264,1046),(292,1047),(291,1082),(263,1081)

"5/28/24"
bounds: (346,1022),(570,1018),(571,1065),(347,1069)

","
bounds: (579,1018),(596,1018),(597,1064),(580,1064)

"7:10"
bounds: (637,1017),(767,1014),(768,1060),(638,1063)

"PM"
bounds: (797,1014),(861,1013),(862,1059),(798,1060)

"Server"
bounds: (349,1077),(541,1076),(541,1120),(349,1121)

":"
bounds: (548,1076),(566,1076),(566,1119),(548,1119)

"Salvador"
bounds: (606,1075),(864,1073),(864,1117),(606,1119)

"A"
bounds: (925,1073),(960,1073),(960,1116),(925,1116)

"S"
bounds: (258,1106),(288,1106),(288,1141),(258,1141)

"D"
bounds: (253,1166),(280,1166),(279,1205),(252,1205)

":"
bounds: (281,1166),(293,1166),(292,1205),(280,1205)

"Dining"
bounds: (349,1137),(539,1134),(540,1185),(350,1188)

"Table"
bounds: (574,1134),(738,1132),(739,1183),(575,1185)

"41"
bounds: (764,1132),(829,1131),(830,1181),(765,1182)

"Seat"
bounds: (349,1201),(474,1201),(474,1238),(349,1238)

"1"
bounds: (513,1201),(542,1201),(542,1238),(513,1238)

"Se"
bounds: (243,1226),(297,1227),(296,1263),(242,1262)

"Ir"
bounds: (243,1285),(295,1286),(294,1325),(242,1324)

"Invoice"
bounds: (350,1257),(575,1256),(575,1301),(350,1302)

":"
bounds: (583,1256),(599,1256),(599,1300),(583,1300)

"240528-03-17"
bounds: (641,1255),(1020,1253),(1020,1298),(641,1300)

"Ticket"
bounds: (1249,1015),(1435,1015),(1435,1057),(1249,1057)

":"
bounds: (1445,1015),(1462,1015),(1462,1057),(1445,1057)

"B17"
bounds: (1500,1015),(1595,1015),(1595,1057),(1500,1057)

"Credit"
bounds: (350,1438),(542,1438),(542,1481),(350,1481)

"Sale"
bounds: (577,1438),(708,1438),(708,1481),(577,1481)

"1"
bounds: (255,1469),(286,1469),(286,1501),(255,1501)

"Status"
bounds: (349,1503),(545,1502),(545,1545),(349,1546)

":"
bounds: (554,1503),(576,1503),(576,1545),(554,1545)

"000000"
bounds: (904,1493),(1097,1493),(1097,1548),(904,1548)

"-"
bounds: (1127,1493),(1159,1493),(1159,1548),(1127,1548)

"Approved"
bounds: (1188,1493),(1447,1493),(1447,1548),(1188,1548)

"1"
bounds: (254,1529),(281,1530),(280,1568),(253,1567)

"1"
bounds: (247,1592),(275,1592),(275,1627),(247,1627)

"Card"
bounds: (351,1625),(487,1625),(487,1674),(351,1674)

"Type"
bounds: (511,1625),(648,1625),(648,1674),(511,1674)

":"
bounds: (653,1625),(675,1625),(675,1674),(653,1674)

"M"
bounds: (903,1623),(935,1622),(936,1663),(904,1664)

"/"
bounds: (938,1622),(966,1621),(967,1662),(939,1663)

"C"
bounds: (967,1621),(999,1620),(1000,1661),(968,1662)

"Card"
bounds: (352,1685),(483,1685),(483,1726),(352,1726)

"Number"
bounds: (515,1685),(712,1685),(712,1726),(515,1726)

":"
bounds: (718,1685),(734,1685),(734,1726),(718,1726)

"XXXXX"
bounds: (1107,1680),(1258,1678),(1258,1717),(1107,1719)

"1"
bounds: (237,1714),(266,1715),(265,1751),(236,1750)

"Card"
bounds: (351,1746),(484,1746),(484,1786),(351,1786)

"Owner"
bounds: (515,1746),(681,1746),(681,1786),(515,1786)

":"
bounds: (687,1746),(703,1746),(703,1786),(687,1786)

"Sub"
bounds: (200,1773),(292,1776),(291,1816),(199,1813)

"SAL"
bounds: (193,1834),(289,1835),(288,1877),(192,1876)

"Entry"
bounds: (350,1809),(515,1805),(516,1853),(351,1857)

"Method"
bounds: (546,1805),(743,1800),(744,1847),(547,1852)

":"
bounds: (750,1800),(766,1800),(767,1847),(751,1847)

"Chip"
bounds: (910,1804),(1032,1805),(1032,1850),(910,1849)

"Auth"
bounds: (347,1866),(485,1866),(485,1909),(347,1909)

"Code"
bounds: (517,1866),(651,1866),(651,1909),(517,1909)

":"
bounds: (654,1866),(675,1866),(675,1909),(654,1909)

"028723"
bounds: (909,1861),(1097,1860),(1097,1899),(909,1900)

"Tot"
bounds: (185,1949),(286,1965),(277,2019),(177,2003)

"AID"
bounds: (350,1975),(455,1991),(446,2046),(342,2029)

":"
bounds: (457,1992),(479,1995),(470,2049),(449,2045)

"TC"
bounds: (349,2050),(414,2051),(413,2092),(348,2091)

":"
bounds: (422,2052),(441,2052),(440,2092),(421,2092)

"APPLAB"
bounds: (347,1931),(549,1931),(549,1973),(347,1973)

":"
bounds: (558,1931),(579,1931),(579,1973),(558,1973)

"MASTERCARD"
bounds: (906,1923),(1229,1919),(1229,1960),(907,1964)

"A0000000041010"
bounds: (907,1979),(1360,1975),(1360,2019),(907,2023)

"W"
bounds: (203,2282),(269,2282),(269,2335),(203,2335)

"18"
bounds: (349,2232),(412,2231),(412,2283),(349,2284)

"%"
bounds: (409,2232),(449,2232),(449,2283),(409,2283)

"Gratuity"
bounds: (479,2231),(748,2229),(748,2281),(479,2283)

"AMOUNT"
bounds: (344,2363),(545,2363),(545,2407),(344,2407)

"ADDITIONAL"
bounds: (341,2489),(678,2482),(679,2528),(342,2535)

"TIP"
bounds: (709,2481),(811,2479),(812,2525),(710,2527)

"TOTAL"
bounds: (342,2556),(511,2556),(511,2599),(342,2599)

"8.91"
bounds: (1499,2219),(1633,2218),(1633,2261),(1499,2262)

"63.10"
bounds: (1472,2342),(1634,2342),(1634,2386),(1472,2386)

"Sign"
bounds: (370,2814),(505,2813),(505,2866),(370,2867)

"X_"
bounds: (536,2814),(635,2814),(635,2866),(536,2866)

"I"
bounds: (385,2942),(424,2941),(425,2999),(386,3000)

"agree"
bounds: (449,2940),(628,2936),(629,2994),(450,2998)

"to"
bounds: (658,2935),(727,2933),(728,2992),(659,2994)

"pay"
bounds: (760,2933),(866,2931),(867,2989),(761,2991)

"the"
bounds: (894,2930),(997,2928),(998,2986),(895,2988)

"total"
bounds: (1031,2927),(1197,2923),(1198,2981),(1032,2985)

"amount"
bounds: (1229,2922),(1433,2917),(1434,2976),(1230,2981)

"above"
bounds: (1466,2917),(1634,2913),(1635,2972),(1467,2976)

"according"
bounds: (352,2999),(658,2994),(659,3059),(353,3064)

"to"
bounds: (694,2993),(764,2992),(765,3057),(695,3058)

"the"
bounds: (792,2991),(898,2989),(899,3054),(793,3056)

"card"
bounds: (933,2989),(1063,2987),(1064,3052),(934,3054)

"issuer"
bounds: (1097,2986),(1301,2982),(1302,3047),(1098,3051)

"agreement"
bounds: (1335,2982),(1635,2977),(1636,3041),(1336,3046)

"."
bounds: (1642,2977),(1659,2977),(1660,3041),(1643,3041)

"Duplicate"
bounds: (778,3195),(1087,3191),(1088,3247),(779,3251)

"Copy"
bounds: (1116,3190),(1260,3188),(1261,3244),(1117,3246)

"WE'LL"
bounds: (390,3408),(646,3404),(647,3470),(391,3474)

"BE"
bounds: (677,3404),(791,3402),(792,3467),(678,3469)

"SEEING"
bounds: (818,3401),(1130,3396),(1131,3462),(819,3467)

"YOU"
bounds: (1159,3396),(1342,3393),(1343,3458),(1160,3461)

"SOON"
bounds: (1368,3393),(1609,3389),(1610,3454),(1369,3458)

"!"
bounds: (1615,3389),(1644,3389),(1645,3454),(1616,3454)

"TERMINAL"
bounds: (712,3494),(1002,3488),(1003,3535),(713,3541)

"3"
bounds: (1018,3488),(1050,3487),(1051,3533),(1019,3534)

"/"
bounds: (1062,3487),(1090,3486),(1091,3532),(1063,3533)

"GATE"
bounds: (1098,3486),(1250,3483),(1251,3530),(1099,3533)

"36"
bounds: (1267,3483),(1332,3482),(1333,3529),(1268,3530)

"2024"
bounds: (738,3776),(824,3775),(824,3814),(738,3815)

"Heartland"
bounds: (835,3775),(1009,3773),(1009,3812),(835,3814)

"Payment"
bounds: (1023,3773),(1181,3771),(1181,3810),(1023,3812)

"Systems"
bounds: (1190,3771),(1347,3769),(1347,3807),(1190,3809)

"©"
bounds: (375,3912),(402,3914),(400,3950),(373,3948)

"2024"
bounds: (413,3914),(499,3920),(496,3956),(411,3950)

"Heartland"
bounds: (509,3920),(684,3931),(681,3968),(507,3957)

"Payment"
bounds: (696,3932),(854,3942),(851,3979),(694,3969)

"Systems"
bounds: (864,3943),(1018,3953),(1015,3990),(862,3980)
"""

In [49]:
accounts = ["Checking Account", "Credit Card - Visa", "Cash"]
categories = ["Groceries", "Dining", "Transportation", "Shopping"]
datetime = "2024-03-20 15:30"

In [50]:
def parse_financial_record(input_ocr: str, accounts: list[str], categories: list[str], datetime: str):
    result = structured_llm.invoke([
            ("system", system_prompt),
            ("human", f"input_ocr: {input_ocr}\n Accounts: {accounts}\n Categories: {categories}\n Current DateTime: {datetime}"),
            
        ])
    
    return result # Returns a python dictionary

In [59]:
parse_financial_record(input_ocr, accounts, categories, datetime)

{'description': 'Ludacris Chicken + Beer',
 'type': 'Expense',
 'valid_input': 'valid',
 'category': 'Dining',
 'time': '19:10',
 'account': 'Credit Card - Visa',
 'date': '2024-05-28',
 'amount': 63.1}

In [52]:
result1 = structured_llm.invoke([
        ("system", system_prompt),
        ("human", f"input_ocr: {input_ocr}\n Accounts: {accounts}\n Categories: {categories}\n Current DateTime: {datetime}"),
        
    ])

print(result1)

{'description': 'Ludacris Chicken + Beer', 'account': 'Credit Card - Visa', 'valid_input': 'valid', 'category': 'Dining', 'type': 'Expense', 'time': '19:10', 'date': '2024-05-28', 'amount': 63.1}


In [53]:
resultG3 = structured_llm.invoke([
        ("system", system_prompt),
        ("human", f"input_ocr: {input_ocr}\n Accounts: {accounts}\n Categories: {categories}\n Current DateTime: {datetime}"),
        
    ])

print(resultG3)

{'description': 'Ludacris Chicken and Beer', 'time': '19:10', 'valid_input': 'valid', 'category': 'Dining', 'type': 'Expense', 'account': 'Credit Card - Visa', 'date': '2024-05-28', 'amount': 63.1}


In [54]:
resultGF = structured_llm.invoke([
        ("system", system_prompt),
        ("human", f"input_ocr: {input_ocr}\n Accounts: {accounts}\n Categories: {categories}\n Current DateTime: {datetime}"),
        
    ])

print(resultGF)

{'description': 'Ludacris Chicken + Beer', 'type': 'Expense', 'valid_input': 'valid', 'category': 'Dining', 'account': 'Credit Card - Visa', 'time': '19:10', 'date': '2024-05-28', 'amount': 63.1}
